# SNU AI Challenge - structured scorer without Transformer

This controlled five-fold experiment uses the new MD5 group-stratified folds and changes only the full-sequence verifier: `verifier="none"`. The matching Transformer baseline must be rerun with the same group split; legacy EXP-003 fold metrics are not directly comparable. The original 9,535 rows, frozen feature cache, resident candidate plans, batch sizes, optimizer, seed, and early-stopping policy remain unchanged. No augmented data is used.

## 1. Parameters

In [1]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
BUNDLE_PATH = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
CODE_BUNDLE_PATH = DRIVE_ROOT / "snu_colab_no_transformer_training_code.tar"
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
GROUP_FOLDS_CSV = WORKDIR / "snuaichallenge_data/splits/group_stratified_folds_5.csv"
DRIVE_GROUP_FOLDS_CSV = DRIVE_ROOT / "splits/group_stratified_folds_5.csv"
DRIVE_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
DRIVE_RESULTS_ROOT = DRIVE_ROOT / "results/full_24class_group_stratified_no_transformer_v1"
LOCAL_CACHE_ROOT = WORKDIR / "feature_cache/full_train_9535"

FOLDS_TO_RUN = [0, 1, 2, 3, 4]
TRAIN_FINAL_SCORER = True
GPU_CONFIG_CONFIRMED = True  # Review the detected GPU cell, then change to True.

HIDDEN_DIM = 256
VERIFIER = "none"
BATCH_SIZE = 128
EVALUATION_BATCH_SIZE = 256
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TOP_H = 5
TOP_M = 8
ALPHA, BETA, GAMMA, DELTA = 0.1, 1.0, 1.0, 1.0
SEED = 42

assert FOLDS_TO_RUN and all(fold in range(5) for fold in FOLDS_TO_RUN)
assert len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))
assert VERIFIER == "none"
assert BATCH_SIZE == 128 and EVALUATION_BATCH_SIZE == 256

## 2. Mount Drive and refresh the no-Transformer code overlay

In [2]:
import importlib
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
else:
    print("google.colab is unavailable; configured paths will be used directly.")

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Training bundle not found: {BUNDLE_PATH}")
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"No-Transformer code bundle not found: {CODE_BUNDLE_PATH}")

WORKDIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    if not TRAIN_CSV.is_file() or not IMAGE_ROOT.is_dir():
        subprocess.run(["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR)], check=True)
elif not TRAIN_CSV.is_file():
    subprocess.run(
        ["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR),
         "snuaichallenge_data/train.csv"],
        check=True,
    )
subprocess.run(
    ["tar", "-xf", str(CODE_BUNDLE_PATH), "-C", str(WORKDIR)],
    check=True,
)
for module_name in list(sys.modules):
    if module_name == "snu_ordering" or module_name.startswith("snu_ordering."):
        del sys.modules[module_name]
importlib.invalidate_caches()
sys.path.insert(0, str(WORKDIR))

print(f"Project root: {WORKDIR}")
print(f"Feature shards: {DRIVE_CACHE_SHARDS}")
print(f"Experiment results: {DRIVE_RESULTS_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/snu_ai
Feature shards: /content/drive/MyDrive/snu ai/cache/full_train_9535_shards
Experiment results: /content/drive/MyDrive/snu ai/results/full_24class_group_stratified_no_transformer_v1


## 3. Build the MD5 group-stratified split and validate GPU/data

The experiment keeps the established batch sizes for a controlled comparison against the new group-stratified Transformer baseline. Review the detected GPU and VRAM below. Training is blocked until `GPU_CONFIG_CONFIRMED=True` is set in the parameter cell.

In [3]:
import json
import random
from collections import Counter

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")

from snu_ordering.data import load_samples

import ast
import hashlib
import json
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
GROUP_SPLIT_SEED = 42
GROUP_SPLIT_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3
GROUP_SPLIT_METHOD = "md5_stratified_group_kfold_v1"


def _compute_md5(path):
    with path.open("rb") as file:
        return hashlib.md5(file.read()).hexdigest()


def _build_md5_group_stratified_folds(train_csv, image_root):
    train = pd.read_csv(train_csv)
    train["answer_tuple"] = train["Answer"].map(
        lambda value: str(ast.literal_eval(value))
    )

    image_records = []
    for _, row in train.iterrows():
        for image_index in range(1, 5):
            image_records.append(
                {
                    "sample_id": row["Id"],
                    "path": image_root / str(row["Id"]) / str(row[f"Input_{image_index}"]),
                }
            )

    print(f"Computing MD5 for {len(image_records)} training images...")
    with ThreadPoolExecutor(max_workers=32) as executor:
        hashes = list(executor.map(_compute_md5, [record["path"] for record in image_records]))
    for record, image_hash in zip(image_records, hashes, strict=True):
        record["hash"] = image_hash

    hash_counter = defaultdict(set)
    for record in image_records:
        hash_counter[record["hash"]].add(record["sample_id"])
    hash_to_samples = {
        image_hash: sample_ids
        for image_hash, sample_ids in hash_counter.items()
        if len(sample_ids) <= MAX_SAMPLES_PER_HASH
    }

    graph = nx.Graph()
    graph.add_nodes_from(train["Id"].unique())
    for sample_ids in hash_to_samples.values():
        sample_ids = list(sample_ids)
        for left in range(len(sample_ids)):
            for right in range(left + 1, len(sample_ids)):
                graph.add_edge(sample_ids[left], sample_ids[right])

    components = list(nx.connected_components(graph))
    id_to_cluster = {
        sample_id: cluster_id
        for cluster_id, component in enumerate(components)
        for sample_id in component
    }
    train["cluster_id"] = train["Id"].map(id_to_cluster)

    splitter = StratifiedGroupKFold(
        n_splits=GROUP_SPLIT_FOLDS,
        shuffle=True,
        random_state=GROUP_SPLIT_SEED,
    )
    train["Fold"] = -1
    for fold, (_, validation_indices) in enumerate(
        splitter.split(
            train,
            train["answer_tuple"],
            groups=train["cluster_id"],
        )
    ):
        train.loc[validation_indices, "Fold"] = fold

    train["split_method"] = GROUP_SPLIT_METHOD
    train["split_seed"] = GROUP_SPLIT_SEED
    train["split_max_samples_per_hash"] = MAX_SAMPLES_PER_HASH
    train["split_sklearn_version"] = sklearn.__version__
    build_summary = {
        "clusters": len(components),
        "excluded_frequent_hashes": sum(
            len(sample_ids) > MAX_SAMPLES_PER_HASH
            for sample_ids in hash_counter.values()
        ),
    }
    return train, build_summary


def _validate_md5_group_stratified_folds(folded, raw_train):
    required = {
        "Id", "Answer", "No_ordering", "answer_tuple", "cluster_id", "Fold",
        "split_method", "split_seed", "split_max_samples_per_hash",
        "split_sklearn_version",
    }
    missing = required.difference(folded.columns)
    if missing:
        raise ValueError(f"Group split is missing columns: {sorted(missing)}")
    if folded["Id"].astype(str).tolist() != raw_train["Id"].astype(str).tolist():
        raise ValueError("Group split IDs/order do not match train.csv")
    if folded["Id"].duplicated().any():
        raise ValueError("Group split contains duplicate IDs")

    expected_answers = raw_train["Answer"].map(lambda value: str(ast.literal_eval(value)))
    if folded["answer_tuple"].astype(str).tolist() != expected_answers.tolist():
        raise ValueError("Group split Answer labels do not match train.csv")
    if set(folded["split_method"].astype(str)) != {GROUP_SPLIT_METHOD}:
        raise ValueError("Unexpected group split method")
    if set(folded["split_seed"].astype(int)) != {GROUP_SPLIT_SEED}:
        raise ValueError("Unexpected group split seed")
    if set(folded["split_max_samples_per_hash"].astype(int)) != {MAX_SAMPLES_PER_HASH}:
        raise ValueError("Unexpected MD5 reuse threshold")

    folded["Fold"] = folded["Fold"].astype(int)
    if set(folded["Fold"]) != set(range(GROUP_SPLIT_FOLDS)):
        raise ValueError("Expected folds 0 through 4")
    cross_fold_clusters = int(
        (folded.groupby("cluster_id")["Fold"].nunique() > 1).sum()
    )
    if cross_fold_clusters:
        raise ValueError(f"{cross_fold_clusters} MD5 clusters cross folds")

    identity_mask = folded["answer_tuple"].eq("[1, 2, 3, 4]")
    no_ordering_mask = folded["No_ordering"].astype(str).str.lower().eq("true")
    if not identity_mask.equals(no_ordering_mask):
        raise ValueError("Identity labels and No_ordering=True rows differ")
    classes_per_fold = folded.groupby("Fold")["answer_tuple"].nunique().to_dict()
    if set(classes_per_fold.values()) != {24}:
        raise ValueError(f"Every fold must contain all 24 classes: {classes_per_fold}")

    return {
        "method": GROUP_SPLIT_METHOD,
        "rows": len(folded),
        "classes": folded["answer_tuple"].nunique(),
        "identity_rows": int(identity_mask.sum()),
        "clusters": folded["cluster_id"].nunique(),
        "cross_fold_clusters": cross_fold_clusters,
        "fold_counts": {
            str(fold): int(count)
            for fold, count in folded["Fold"].value_counts().sort_index().items()
        },
        "built_with_sklearn": str(folded["split_sklearn_version"].iloc[0]),
    }


GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
DRIVE_GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_GROUP_FOLDS_CSV.is_file():
    shutil.copy2(DRIVE_GROUP_FOLDS_CSV, GROUP_FOLDS_CSV)
elif not GROUP_FOLDS_CSV.is_file():
    folded_frame, group_split_build = _build_md5_group_stratified_folds(
        TRAIN_CSV,
        IMAGE_ROOT,
    )
    temporary_csv = GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    folded_frame.to_csv(temporary_csv, index=False)
    temporary_csv.replace(GROUP_FOLDS_CSV)
    print("group split build:", json.dumps(group_split_build, indent=2))

raw_train_frame = pd.read_csv(TRAIN_CSV)
group_fold_frame = pd.read_csv(GROUP_FOLDS_CSV)
group_split_summary = _validate_md5_group_stratified_folds(
    group_fold_frame,
    raw_train_frame,
)
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    temporary_drive_csv = DRIVE_GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    shutil.copy2(GROUP_FOLDS_CSV, temporary_drive_csv)
    temporary_drive_csv.replace(DRIVE_GROUP_FOLDS_CSV)
print("group split:", json.dumps(group_split_summary, indent=2))

samples = load_samples(
    GROUP_FOLDS_CSV,
    WORKDIR / "snuaichallenge_data/train",
    require_labels=True,
    validate_images=False,
)
answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
identity_count = answer_counts[(1, 2, 3, 4)]
assert len(samples) == 9535
assert len(answer_counts) == 24
assert identity_count == 1478
assert set(fold_counts) == set(range(5))
assert sum(fold_counts.values()) == 9535
assert all(sample.fold is not None for sample in samples)

gpu_properties = torch.cuda.get_device_properties(0)
gpu_review = {
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gib": gpu_properties.total_memory / 1024**3,
    "cuda_capability": torch.cuda.get_device_capability(0),
    "precision": "BF16 AMP" if torch.cuda.get_device_capability(0)[0] >= 8 else "FP16 AMP",
    "train_batch_size": BATCH_SIZE,
    "evaluation_batch_size": EVALUATION_BATCH_SIZE,
    "resident_original_samples": len(samples),
    "verifier": VERIFIER,
}
print(json.dumps(gpu_review, indent=2))
if not GPU_CONFIG_CONFIRMED:
    raise RuntimeError(
        "Review GPU/VRAM/batch settings above, then set GPU_CONFIG_CONFIRMED=True."
    )

group split: {
  "method": "md5_stratified_group_kfold_v1",
  "rows": 9535,
  "classes": 24,
  "identity_rows": 1478,
  "clusters": 8147,
  "cross_fold_clusters": 0,
  "fold_counts": {
    "0": 1915,
    "1": 1905,
    "2": 1886,
    "3": 1911,
    "4": 1918
  },
  "built_with_sklearn": "1.6.1"
}
{
  "torch": "2.11.0+cu128",
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "vram_gib": 94.97076416015625,
  "cuda_capability": [
    12,
    0
  ],
  "precision": "BF16 AMP",
  "train_batch_size": 128,
  "evaluation_batch_size": 256,
  "resident_original_samples": 9535,
  "verifier": "none"
}


## 4. Restore the completed original feature cache

In [4]:
import tarfile
import time

from snu_ordering.qwen_representations import cache_path_for

def format_duration(seconds):
    seconds = max(0, int(round(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cache_shards = sorted(DRIVE_CACHE_SHARDS.glob("cache_*.tar"))
if not cache_shards:
    raise FileNotFoundError(f"No original feature-cache shards found: {DRIVE_CACHE_SHARDS}")
restore_started = time.monotonic()
for shard_index, shard_path in enumerate(cache_shards, 1):
    with tarfile.open(shard_path, "r") as archive:
        archive.extractall(LOCAL_CACHE_ROOT)
    elapsed = time.monotonic() - restore_started
    rate = shard_index / elapsed if elapsed else 0.0
    remaining = len(cache_shards) - shard_index
    eta = remaining / rate if rate else 0.0
    print(
        f"cache shards {shard_index}/{len(cache_shards)} | remaining={remaining} | "
        f"elapsed={format_duration(elapsed)} | ETA={format_duration(eta)}",
        flush=True,
    )
missing_cache = [
    sample.sample_id for sample in samples
    if not cache_path_for(LOCAL_CACHE_ROOT, sample.sample_id).is_file()
]
if missing_cache:
    raise FileNotFoundError(
        f"Missing {len(missing_cache)} original cached representations; first IDs: {missing_cache[:5]}"
    )
print(f"original feature cache ready: {len(samples)}/{len(samples)}")

cache shards 1/48 | remaining=47 | elapsed=00:00:00 | ETA=00:00:01
cache shards 2/48 | remaining=46 | elapsed=00:00:00 | ETA=00:00:01
cache shards 3/48 | remaining=45 | elapsed=00:00:00 | ETA=00:00:01
cache shards 4/48 | remaining=44 | elapsed=00:00:00 | ETA=00:00:01
cache shards 5/48 | remaining=43 | elapsed=00:00:00 | ETA=00:00:01
cache shards 6/48 | remaining=42 | elapsed=00:00:00 | ETA=00:00:01
cache shards 7/48 | remaining=41 | elapsed=00:00:00 | ETA=00:00:01
cache shards 8/48 | remaining=40 | elapsed=00:00:00 | ETA=00:00:01
cache shards 9/48 | remaining=39 | elapsed=00:00:00 | ETA=00:00:01
cache shards 10/48 | remaining=38 | elapsed=00:00:00 | ETA=00:00:01


/tmp/ipykernel_7979/782362471.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(LOCAL_CACHE_ROOT)


cache shards 11/48 | remaining=37 | elapsed=00:00:00 | ETA=00:00:01
cache shards 12/48 | remaining=36 | elapsed=00:00:00 | ETA=00:00:01
cache shards 13/48 | remaining=35 | elapsed=00:00:00 | ETA=00:00:01
cache shards 14/48 | remaining=34 | elapsed=00:00:00 | ETA=00:00:01
cache shards 15/48 | remaining=33 | elapsed=00:00:00 | ETA=00:00:01
cache shards 16/48 | remaining=32 | elapsed=00:00:00 | ETA=00:00:01
cache shards 17/48 | remaining=31 | elapsed=00:00:00 | ETA=00:00:01
cache shards 18/48 | remaining=30 | elapsed=00:00:00 | ETA=00:00:01
cache shards 19/48 | remaining=29 | elapsed=00:00:00 | ETA=00:00:01
cache shards 20/48 | remaining=28 | elapsed=00:00:00 | ETA=00:00:01
cache shards 21/48 | remaining=27 | elapsed=00:00:00 | ETA=00:00:01
cache shards 22/48 | remaining=26 | elapsed=00:00:00 | ETA=00:00:01
cache shards 23/48 | remaining=25 | elapsed=00:00:01 | ETA=00:00:01
cache shards 24/48 | remaining=24 | elapsed=00:00:01 | ETA=00:00:01
cache shards 25/48 | remaining=23 | elapsed=00:0

## 5. Prepare the same GPU-resident features and grouping plans

In [5]:
import gc

from snu_ordering.structured_scorer import LightweightStructuredScorer
from snu_ordering.structured_training import (
    StructuredTrainingConfig,
    prepare_structured_samples,
    train_prepared_structured_scorer,
)

DEVICE = torch.device("cuda")
torch.set_float32_matmul_precision("high")
training_config = StructuredTrainingConfig(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    evaluation_batch_size=EVALUATION_BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=SEED,
    top_h=TOP_H,
    top_m=TOP_M,
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA,
    delta=DELTA,
    use_amp=True,
)
prepared_samples = prepare_structured_samples(
    samples, LOCAL_CACHE_ROOT, DEVICE, top_h=TOP_H
)
assert len(prepared_samples) == 9535

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_no_transformer_scorer():
    scorer = LightweightStructuredScorer(
        hidden_dim=HIDDEN_DIM, verifier=VERIFIER
    ).to(DEVICE)
    assert scorer.verifier_type == "none"
    assert scorer.sequence_transformer is None
    assert scorer.sequence_head is None
    assert scorer.cls_token is None
    assert scorer.position_embedding is None
    return scorer

probe_scorer = make_no_transformer_scorer()
print("trainable parameters without Transformer:", sum(
    parameter.numel() for parameter in probe_scorer.parameters() if parameter.requires_grad
))
del probe_scorer
gc.collect()
torch.cuda.empty_cache()

prepared candidates: 500/9535
prepared candidates: 1000/9535
prepared candidates: 1500/9535
prepared candidates: 2000/9535
prepared candidates: 2500/9535
prepared candidates: 3000/9535
prepared candidates: 3500/9535
prepared candidates: 4000/9535
prepared candidates: 4500/9535
prepared candidates: 5000/9535
prepared candidates: 5500/9535
prepared candidates: 6000/9535
prepared candidates: 6500/9535
prepared candidates: 7000/9535
prepared candidates: 7500/9535
prepared candidates: 8000/9535
prepared candidates: 8500/9535
prepared candidates: 9000/9535
prepared candidates: 9500/9535
prepared candidates: 9535/9535
resident training data: features=1.13 GiB, candidate_plans=0.35 GiB on cuda
trainable parameters without Transformer: 2166274


## 6. Train the selected folds

In [6]:
def run_fold(fold):
    output_dir = DRIVE_RESULTS_ROOT / f"fold_{fold}"
    if (output_dir / "training_summary.json").is_file():
        print(f"fold {fold}: already completed, skipping")
        return
    train_samples = [item for item in prepared_samples if item.sample.fold != fold]
    validation_samples = [item for item in prepared_samples if item.sample.fold == fold]
    assert {item.sample.rank_label for item in train_samples} == set(answer_counts)
    assert {item.sample.rank_label for item in validation_samples} == set(answer_counts)
    seed_everything(SEED)
    scorer = make_no_transformer_scorer()
    print(
        f"fold {fold}: train={len(train_samples)}, validation={len(validation_samples)}, "
        f"batch={BATCH_SIZE}, eval_batch={EVALUATION_BATCH_SIZE}, verifier={VERIFIER}"
    )
    train_prepared_structured_scorer(
        scorer, train_samples, validation_samples, output_dir, training_config
    )
    del scorer
    gc.collect()
    torch.cuda.empty_cache()

for fold in FOLDS_TO_RUN:
    run_fold(fold)

fold 0: train=7620, validation=1915, batch=128, eval_batch=256, verifier=none


/content/snu_ai/snu_ordering/structured_training.py:376: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_fp16_scaler)


{"epoch": 1, "train_loss": 2.8870193515236924, "samples": 1915, "exact_match_accuracy": 0.15509138381201043, "lightweight_top8_recall": 0.6652741514360313}
{"epoch": 2, "train_loss": 2.6712907746082215, "samples": 1915, "exact_match_accuracy": 0.21671018276762402, "lightweight_top8_recall": 0.7002610966057441}
{"epoch": 3, "train_loss": 2.4377347629527093, "samples": 1915, "exact_match_accuracy": 0.21618798955613577, "lightweight_top8_recall": 0.7086161879895562}
{"epoch": 4, "train_loss": 2.269653829063956, "samples": 1915, "exact_match_accuracy": 0.2349869451697128, "lightweight_top8_recall": 0.7075718015665796}
{"epoch": 5, "train_loss": 2.110126186355831, "samples": 1915, "exact_match_accuracy": 0.22663185378590078, "lightweight_top8_recall": 0.6924281984334204}
{"epoch": 6, "train_loss": 1.9796355836973416, "samples": 1915, "exact_match_accuracy": 0.2558746736292428, "lightweight_top8_recall": 0.6976501305483028}
{"epoch": 7, "train_loss": 1.8499754554017636, "samples": 1915, "exa

## 7. Aggregate five-fold results

In [7]:
import statistics

fold_records = []
for fold in range(5):
    summary_path = DRIVE_RESULTS_ROOT / f"fold_{fold}/training_summary.json"
    if not summary_path.is_file():
        continue
    with summary_path.open("r", encoding="utf-8") as file:
        summary = json.load(file)
    best = max(summary["history"], key=lambda row: row["exact_match_accuracy"])
    fold_records.append({
        "fold": fold,
        "best_epoch": best["epoch"],
        "samples": best["samples"],
        "exact_match_accuracy": best["exact_match_accuracy"],
        "lightweight_top8_recall": best.get("lightweight_top8_recall"),
        "epochs_completed": summary["epochs_completed"],
        "verifier": summary["config"].get("verifier", VERIFIER),
    })

aggregate = {
    "experiment": "full_24class_group_stratified_no_transformer_v1",
    "verifier": VERIFIER,
    "folds": fold_records,
    "completed_fold_count": len(fold_records),
}
if len(fold_records) == 5:
    total_validation_samples = sum(row["samples"] for row in fold_records)
    aggregate["total_validation_samples"] = total_validation_samples
    aggregate["weighted_exact_match_accuracy"] = sum(
        row["samples"] * row["exact_match_accuracy"] for row in fold_records
    ) / total_validation_samples
    selected_final_epochs = max(
        1, int(round(statistics.median(row["best_epoch"] for row in fold_records)))
    )
    aggregate["selected_final_epochs"] = selected_final_epochs
else:
    selected_final_epochs = None
with (DRIVE_RESULTS_ROOT / "five_fold_summary.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(aggregate, file, indent=2)
    file.write("\n")
print(json.dumps(aggregate, indent=2))

{
  "experiment": "full_24class_group_stratified_no_transformer_v1",
  "verifier": "none",
  "folds": [
    {
      "fold": 0,
      "best_epoch": 9,
      "samples": 1915,
      "exact_match_accuracy": 0.2699738903394256,
      "lightweight_top8_recall": 0.7002610966057441,
      "epochs_completed": 13,
      "verifier": "none"
    },
    {
      "fold": 1,
      "best_epoch": 13,
      "samples": 1905,
      "exact_match_accuracy": 0.28346456692913385,
      "lightweight_top8_recall": 0.7186351706036745,
      "epochs_completed": 17,
      "verifier": "none"
    },
    {
      "fold": 2,
      "best_epoch": 16,
      "samples": 1886,
      "exact_match_accuracy": 0.28950159066808057,
      "lightweight_top8_recall": 0.7067868504772005,
      "epochs_completed": 20,
      "verifier": "none"
    },
    {
      "fold": 3,
      "best_epoch": 15,
      "samples": 1911,
      "exact_match_accuracy": 0.28885400313971743,
      "lightweight_top8_recall": 0.7221350078492935,
      "epochs_co

## 8. Train one final no-Transformer scorer

In [8]:
from dataclasses import replace

if TRAIN_FINAL_SCORER:
    if selected_final_epochs is None:
        raise RuntimeError("All five folds must finish before final training.")
    final_output_dir = DRIVE_RESULTS_ROOT / "final_scorer"
    final_summary_path = final_output_dir / "training_summary.json"
    if final_summary_path.is_file():
        print(f"final scorer already completed: {final_output_dir}")
    else:
        seed_everything(SEED)
        final_scorer = make_no_transformer_scorer()
        final_config = replace(
            training_config,
            max_epochs=selected_final_epochs,
            patience=selected_final_epochs,
        )
        print(
            f"final no-Transformer training: samples={len(samples)}, "
            f"epochs={selected_final_epochs}, batch={BATCH_SIZE}"
        )
        train_prepared_structured_scorer(
            final_scorer, prepared_samples, [], final_output_dir, final_config
        )
        del final_scorer
        gc.collect()
        torch.cuda.empty_cache()

final no-Transformer training: samples=9535, epochs=15, batch=128
{"epoch": 1, "train_loss": 2.8492889543822377}
{"epoch": 2, "train_loss": 2.607035164090433}
{"epoch": 3, "train_loss": 2.400863570121801}
{"epoch": 4, "train_loss": 2.241868088316655}
{"epoch": 5, "train_loss": 2.0996586391296947}
{"epoch": 6, "train_loss": 1.9716228826669855}
{"epoch": 7, "train_loss": 1.8576817653537985}
{"epoch": 8, "train_loss": 1.7513160255359115}
{"epoch": 9, "train_loss": 1.6437054393675294}
{"epoch": 10, "train_loss": 1.5490835851690314}
{"epoch": 11, "train_loss": 1.444174617050703}
{"epoch": 12, "train_loss": 1.3567959687318487}
{"epoch": 13, "train_loss": 1.29279388650025}
{"epoch": 14, "train_loss": 1.2032981454357152}
{"epoch": 15, "train_loss": 1.1504078182801565}


## 9. Expected outputs

- `results/full_24class_group_stratified_no_transformer_v1/fold_*/best.pt`
- `results/full_24class_group_stratified_no_transformer_v1/five_fold_summary.json`
- `results/full_24class_group_stratified_no_transformer_v1/final_scorer/final.pt`

Compare the weighted five-fold Exact Match only against `results/full_24class_group_stratified_v1/five_fold_summary.json`, not legacy EXP-003. Do not combine fold checkpoints. Record measured results in `MODELS.md` only after all five folds complete.